# 🛰️ Notebook 1 — YOLO11n / YOLO11s / YOLO11m Training + SAHI
## Satellite Aerial Object Detection — Conference Paper Pipeline
### Dataset: COCO format · Classes: plane, ship, vehicle

> **Kaggle GPU:** Settings → Accelerator → GPU T4 x2 → Save → Restart  
> GPU is strongly recommended for training. The notebook auto-falls back to CPU if GPU is unavailable.  
> **Cell order:** Cell 1 (Setup + Full EDA) → Cell 2 (Preprocessing + Hyperparams) → Cell 3 (Train + SAHI)

## 📦 Cell 1 — Setup · Device Detection · Full EDA on Raw Dataset

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP + DEVICE + FULL EDA (MEMORY-SAFE)
# Each figure is saved then immediately closed (plt.close) to prevent
# the browser from crashing on low-RAM Kaggle sessions.
# ═══════════════════════════════════════════════════════════════════════

# ── 1A. Install ─────────────────────────────────────────────────────
!pip install ultralytics sahi seaborn scipy opencv-python-headless -q

import os, json, shutil, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # ← non-interactive backend — avoids browser OOM
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from collections import Counter
import torch

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams.update({"font.size": 10, "axes.titlesize": 11,
                     "axes.titleweight": "bold"})

def save_close(fig, path, dpi=150):
    """Save figure, close it immediately, and free memory."""
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    gc.collect()
    print(f"   ✅ Saved → {path}")

# ── 1B. Device detection ────────────────────────────────────────────
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("=" * 65)
print("DEVICE CONFIGURATION")
print("=" * 65)
if torch.cuda.is_available():
    print(f"  ✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"     VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"     CUDA: {torch.version.cuda}")
else:
    print("  ⚠️  No GPU — CPU mode active. Training will be much slower.")
    print("      Recommendation: Kaggle → Settings → Accelerator → GPU T4 x2")
print(f"  Active device: {device}")

# ── 1C. Paths ───────────────────────────────────────────────────────
DATASET_PATH  = "/kaggle/input/datasets/rifat4018/satelite-coco"
WORK_DIR      = "/kaggle/working"
SAVE_DIR      = f"{WORK_DIR}/saved_models"
PLOT_DIR      = f"{WORK_DIR}/plots"
TRAIN_IMG_DIR = f"{DATASET_PATH}/train"

for d in [SAVE_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

TRAIN_JSON = f"{DATASET_PATH}/train/_annotations.coco.json"
VAL_JSON   = f"{DATASET_PATH}/valid/_annotations.coco.json"
TEST_JSON  = f"{DATASET_PATH}/test/_annotations.coco.json"

def load_coco(path):
    with open(path) as f:
        return json.load(f)

train_data  = load_coco(TRAIN_JSON)
val_data    = load_coco(VAL_JSON)
test_data   = load_coco(TEST_JSON)
cat_map     = {c["id"]: c["name"] for c in train_data["categories"]}
CLASS_NAMES = list(cat_map.values())   # ['plane', 'ship', 'vehicle']
COLORS      = {"plane": "#2196F3", "ship": "#FF9800", "vehicle": "#4CAF50"}

# ═══════════════════════════════════════════════════════════════════════
# SECTION A — RAW DATASET STATISTICS (printed only — no figure)
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("SECTION A — RAW DATASET STATISTICS")
print("="*65)

splits = {"Train": train_data, "Val": val_data, "Test": test_data}
for sname, d in splits.items():
    n_imgs  = len(d["images"])
    n_anns  = len(d["annotations"])
    n_empty = n_imgs - len(set(a["image_id"] for a in d["annotations"]))
    print(f"  {sname:<6}: {n_imgs:>5} images | {n_anns:>6} annotations | "
          f"{n_empty:>4} empty | {n_anns/max(n_imgs,1):.2f} ann/img avg")
print(f"  Classes: {CLASS_NAMES}")

widths  = [img["width"]  for img in train_data["images"]]
heights = [img["height"] for img in train_data["images"]]
unique_res = set(zip(widths, heights))
print(f"  Resolution (train): {np.mean(widths):.0f}×{np.mean(heights):.0f} avg | "
      f"{len(unique_res)} unique combos")
if len(unique_res) == 1:
    w0, h0 = list(unique_res)[0]
    print(f"  ✅ All images uniform: {w0}×{h0}")

# Precompute everything we need across sections
sizes  = np.array([a["bbox"][2]*a["bbox"][3] for a in train_data["annotations"]])
ratios = [a["bbox"][2]/a["bbox"][3] for a in train_data["annotations"] if a["bbox"][3]>0]
small  = int(np.sum(sizes <  32**2))
medium = int(np.sum((sizes >= 32**2) & (sizes < 96**2)))
large  = int(np.sum(sizes >= 96**2))
total  = len(sizes)

img_obj_count = Counter(a["image_id"] for a in train_data["annotations"])
for img in train_data["images"]:
    if img["id"] not in img_obj_count:
        img_obj_count[img["id"]] = 0
density = list(img_obj_count.values())

train_cnts = Counter(cat_map[a["category_id"]] for a in train_data["annotations"])
val_cnts   = Counter(cat_map[a["category_id"]] for a in val_data["annotations"])
test_cnts  = Counter(cat_map[a["category_id"]] for a in test_data["annotations"])

print(f"  Small (<32²): {small:,} ({100*small/total:.1f}%)  "
      f"Medium: {medium:,} ({100*medium/total:.1f}%)  "
      f"Large: {large:,} ({100*large/total:.1f}%)")
print(f"  Aspect ratio: mean={np.mean(ratios):.3f}, std={np.std(ratios):.3f}")
print(f"  Ann density : mean={np.mean(density):.1f}, max={max(density)}, std={np.std(density):.1f}")

# ═══════════════════════════════════════════════════════════════════════
# SECTION B — CLASS DISTRIBUTION (fig saved + closed immediately)
# ═══════════════════════════════════════════════════════════════════════
print("\nSection B — Class distribution...")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("EDA Fig 1 — Class Distribution per Split", fontweight="bold")
for ax, (sname, d) in zip(axes, splits.items()):
    cnts = Counter(cat_map[a["category_id"]] for a in d["annotations"])
    bars = ax.bar(cnts.keys(), cnts.values(),
                  color=[COLORS.get(k,"#9E9E9E") for k in cnts.keys()],
                  edgecolor="white", linewidth=1.2)
    ax.set_title(f"{sname} Split")
    ax.set_ylabel("Count")
    for bar, (k, v) in zip(bars, cnts.items()):
        ax.text(bar.get_x()+bar.get_width()/2, v + max(cnts.values())*0.01,
                f"{v:,}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_01_class_distribution.png")

# ═══════════════════════════════════════════════════════════════════════
# SECTION C — OBJECT SIZE + ASPECT RATIO
# ═══════════════════════════════════════════════════════════════════════
print("  Section C — Object size & geometry...")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("EDA Fig 2 — Object Size & Geometry (Train)", fontweight="bold")

axes[0].hist(sizes, bins=50, color="#9C27B0", alpha=0.8, edgecolor="white")
axes[0].axvline(32**2, color="red",    linestyle="--", linewidth=1.5,
                label=f"Small (<32²) {100*small/total:.0f}%")
axes[0].axvline(96**2, color="orange", linestyle="--", linewidth=1.5,
                label=f"Med (<96²) {100*medium/total:.0f}%")
axes[0].set_title("BBox Area (px²)")
axes[0].set_xlabel("Area")
axes[0].legend(fontsize=8)

axes[1].pie([small, medium, large],
            labels=[f"Small\n{100*small/total:.1f}%",
                    f"Medium\n{100*medium/total:.1f}%",
                    f"Large\n{100*large/total:.1f}%"],
            colors=["#F44336","#FF9800","#4CAF50"],
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[1].set_title("Scale Breakdown")

axes[2].hist(ratios, bins=40, color="#00BCD4", alpha=0.8, edgecolor="white")
axes[2].axvline(1.0, color="red", linestyle="--", label="Square (1:1)")
axes[2].set_title(f"Aspect Ratio W/H\nmean={np.mean(ratios):.2f} std={np.std(ratios):.2f}")
axes[2].set_xlabel("W/H")
axes[2].legend(fontsize=8)
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_02_size_geometry.png")

# ═══════════════════════════════════════════════════════════════════════
# SECTION D — OBJECT DENSITY PER IMAGE
# ═══════════════════════════════════════════════════════════════════════
print("  Section D — Density per image...")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("EDA Fig 3 — Object Density per Image (Train)", fontweight="bold")

axes[0].hist(density, bins=35, color="#F44336", alpha=0.8, edgecolor="white")
axes[0].set_title(f"Objects/Image  mean={np.mean(density):.1f}, max={max(density)}")
axes[0].set_xlabel("Object Count")

class_density = {}
for cls_name in CLASS_NAMES:
    cls_id = [k for k,v in cat_map.items() if v==cls_name][0]
    per_img = Counter(a["image_id"] for a in train_data["annotations"]
                      if a["category_id"]==cls_id)
    class_density[cls_name] = list(per_img.values())
bp = axes[1].boxplot([class_density[c] for c in CLASS_NAMES],
                     labels=CLASS_NAMES, patch_artist=True)
for patch, color in zip(bp["boxes"], ["#2196F3","#FF9800","#4CAF50"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("Per-Class Count/Image")
axes[1].set_ylabel("Count")
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_03_density.png")

# ═══════════════════════════════════════════════════════════════════════
# SECTION E — SPATIAL HEATMAP (memory-efficient: accumulate, then plot)
# ═══════════════════════════════════════════════════════════════════════
print("  Section E — Spatial heatmap (memory-efficient)...")
W_img = int(np.mean(widths))
H_img = int(np.mean(heights))
# Use float16 to halve memory usage
heatmap_all = np.zeros((H_img, W_img), dtype=np.float16)
heatmaps    = {cls: np.zeros((H_img, W_img), dtype=np.float16) for cls in CLASS_NAMES}

for ann in train_data["annotations"]:
    x, y, w, h = ann["bbox"]
    x1, y1 = max(0, int(x)),          max(0, int(y))
    x2, y2 = min(W_img-1, int(x+w)),  min(H_img-1, int(y+h))
    if x2 > x1 and y2 > y1:
        heatmap_all[y1:y2, x1:x2] += 1
        heatmaps[cat_map[ann["category_id"]]][y1:y2, x1:x2] += 1

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("EDA Fig 4 — BBox Spatial Heatmap (Train)", fontweight="bold")
for ax, (title, hm) in zip(axes,
    [("All Classes", heatmap_all)] + [(c, heatmaps[c]) for c in CLASS_NAMES]):
    im = ax.imshow(np.array(hm, dtype=np.float32),
                   cmap="hot", interpolation="nearest", aspect="auto")
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_04_spatial_heatmap.png")

# Free heatmap arrays immediately
del heatmap_all, heatmaps
gc.collect()

# ═══════════════════════════════════════════════════════════════════════
# SECTION F — ANNOTATED SAMPLE GRID (3×3, max 6 images to save memory)
# ═══════════════════════════════════════════════════════════════════════
print("  Section F — Annotated sample grid...")
COLOR_MAP_CV = {"plane":(255,80,80), "ship":(80,200,80), "vehicle":(80,80,255)}
ann_by_img = {}
for a in train_data["annotations"]:
    ann_by_img.setdefault(a["image_id"], []).append(a)

# Use 6 samples in a 2×3 grid (lighter than 3×3)
sample_imgs = random.sample(train_data["images"], min(6, len(train_data["images"])))
fig, axes   = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("EDA Fig 5 — Train Samples with Ground-Truth Annotations", fontweight="bold")
axes = axes.flatten()

for i, img_info in enumerate(sample_imgs):
    img_path = os.path.join(TRAIN_IMG_DIR, img_info["file_name"])
    img = cv2.imread(img_path)
    if img is None:
        axes[i].text(0.5, 0.5, "Not found", ha="center", va="center",
                     transform=axes[i].transAxes)
        axes[i].axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cnts = {c:0 for c in CLASS_NAMES}
    for ann in ann_by_img.get(img_info["id"], []):
        x, y, w, h = [int(v) for v in ann["bbox"]]
        cls   = cat_map[ann["category_id"]]
        color = COLOR_MAP_CV.get(cls, (255,255,255))
        cv2.rectangle(img, (x,y), (x+w,y+h), color, 2)
        cv2.putText(img, cls, (x, max(0, y-8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        cnts[cls] += 1
    axes[i].imshow(img)
    axes[i].set_title(f"P:{cnts['plane']} S:{cnts['ship']} V:{cnts['vehicle']}  "
                      f"({img_info['width']}×{img_info['height']})", fontsize=9)
    axes[i].axis("off")
    del img   # free immediately
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_05_sample_grid.png", dpi=120)
del ann_by_img
gc.collect()

# ═══════════════════════════════════════════════════════════════════════
# SECTION G — CLASS CO-OCCURRENCE MATRIX
# ═══════════════════════════════════════════════════════════════════════
print("  Section G — Co-occurrence matrix...")
cooccur = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int)
img_class_map = {}
for a in train_data["annotations"]:
    img_class_map.setdefault(a["image_id"], set()).add(cat_map[a["category_id"]])
for cls_set in img_class_map.values():
    for ai, na in enumerate(CLASS_NAMES):
        for bi, nb_ in enumerate(CLASS_NAMES):
            if na in cls_set and nb_ in cls_set:
                cooccur[ai][bi] += 1

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pd.DataFrame(cooccur, index=CLASS_NAMES, columns=CLASS_NAMES),
            annot=True, fmt="d", cmap="Blues", linewidths=0.5, ax=ax)
ax.set_title("EDA Fig 6 — Class Co-occurrence Matrix\n(images containing both classes)",
             fontweight="bold")
plt.tight_layout()
save_close(fig, f"{PLOT_DIR}/eda_06_cooccurrence.png")

# ═══════════════════════════════════════════════════════════════════════
# SECTION H — ANNOTATION QUALITY AUDIT + SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("SECTION H — ANNOTATION QUALITY AUDIT (Raw, before clipping)")
print("="*65)
neg_x  = sum(1 for a in train_data["annotations"] if a["bbox"][0] < 0)
neg_y  = sum(1 for a in train_data["annotations"] if a["bbox"][1] < 0)
zero_w = sum(1 for a in train_data["annotations"] if a["bbox"][2] <= 0)
zero_h = sum(1 for a in train_data["annotations"] if a["bbox"][3] <= 0)
total_issues = neg_x + neg_y + zero_w + zero_h
print(f"  Negative x    : {neg_x}")
print(f"  Negative y    : {neg_y}")
print(f"  Zero-width    : {zero_w}")
print(f"  Zero-height   : {zero_h}")
print(f"  Total issues  : {total_issues}")
if total_issues > 0:
    print("  ⚠️  Fixed in Cell 2 (coords clipped to [0,1])")
else:
    print("  ✅ No annotation issues detected")

print("\n" + "="*65)
print("EDA COMPLETE — FULL SUMMARY TABLE")
print("="*65)
summary_df = pd.DataFrame({
    "Metric": [
        "Train images","Val images","Test images",
        "Train annotations","Val annotations","Test annotations",
        "Classes","Width mean","Height mean","Unique resolutions",
        "plane (train)","ship (train)","vehicle (train)",
        "Small (<32²) %","Medium (32-96) %","Large (>96²) %",
        "Ann/image mean","Ann/image max","Ann/image std",
        "Aspect ratio mean","Aspect ratio std",
        "Annotation quality issues",
    ],
    "Value": [
        len(train_data["images"]), len(val_data["images"]), len(test_data["images"]),
        len(train_data["annotations"]),len(val_data["annotations"]),len(test_data["annotations"]),
        len(CLASS_NAMES), f"{np.mean(widths):.0f}", f"{np.mean(heights):.0f}", len(unique_res),
        train_cnts.get("plane",0), train_cnts.get("ship",0), train_cnts.get("vehicle",0),
        f"{100*small/total:.1f}%", f"{100*medium/total:.1f}%", f"{100*large/total:.1f}%",
        f"{np.mean(density):.2f}", max(density), f"{np.std(density):.2f}",
        f"{np.mean(ratios):.3f}", f"{np.std(ratios):.3f}",
        total_issues,
    ]
})
print(summary_df.to_string(index=False))
summary_df.to_csv(f"{SAVE_DIR}/eda_summary.csv", index=False)
print(f"\n✅ EDA summary saved → {SAVE_DIR}/eda_summary.csv")
print("\n📌 Proceed to Cell 2 — Preprocessing + Hyperparameters")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
DEVICE CONFIGURATION
  ✅ GPU : Tesla T4
     VRAM: 15.6 GB
     CUDA: 12.8
  Active device: cuda:0

SECTION A — RAW DATASET STATISTICS
  Train :  1305 images |  44909 annotations |   30 empty | 34.41 ann/img avg
  Val   :    54 images |   2032 annotations |    1 empty | 37.63 ann/img avg
  Test  :    55 images |   1826 annotations |    1 empty | 33.20 ann/img avg
  Classes: ['objects', 'plane', 'ship', 'vehicle']
  Res

## ⚙️ Cell 2 — Preprocessing: COCO → YOLO + data.yaml + Hyperparameter Config

In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — PREPROCESSING + HYPERPARAMETER CONFIG
# ═══════════════════════════════════════════════════════════════════════

# ── 2A. COCO → YOLO label conversion ────────────────────────────────
YOLO_DIR = f"{WORK_DIR}/dataset"
ID_MAP   = {1: 0, 2: 1, 3: 2}   # plane=0, ship=1, vehicle=2

def convert_coco_to_yolo(json_path, img_src_dir, dest_img_dir, dest_lbl_dir):
    os.makedirs(dest_img_dir, exist_ok=True)
    os.makedirs(dest_lbl_dir, exist_ok=True)
    with open(json_path) as f:
        coco = json.load(f)
    images     = {img["id"]: img for img in coco["images"]}
    ann_by_img = {}
    for ann in coco["annotations"]:
        ann_by_img.setdefault(ann["image_id"], []).append(ann)

    converted = skipped = invalid = 0
    for img_id, img_info in images.items():
        fname = img_info["file_name"]
        src   = os.path.join(img_src_dir, fname)
        if not os.path.exists(src):
            skipped += 1
            continue
        shutil.copy(src, os.path.join(dest_img_dir, fname))
        W, H  = img_info["width"], img_info["height"]
        lines = []
        for ann in ann_by_img.get(img_id, []):
            cid = ID_MAP.get(ann["category_id"])
            if cid is None:
                continue
            x, y, w, h = ann["bbox"]
            cx = np.clip((x + w/2)/W, 0.0, 1.0)
            cy = np.clip((y + h/2)/H, 0.0, 1.0)
            nw = np.clip(w/W,         0.0, 1.0)
            nh = np.clip(h/H,         0.0, 1.0)
            if nw > 1e-4 and nh > 1e-4:
                lines.append(f"{cid} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            else:
                invalid += 1
        lbl_name = os.path.splitext(fname)[0] + ".txt"
        with open(os.path.join(dest_lbl_dir, lbl_name), "w") as f:
            f.write("\n".join(lines))
        converted += 1

    print(f"  ✅ {converted:>5} converted | {skipped:>4} missing | {invalid:>4} degenerate removed")

for split, json_f in [("train", TRAIN_JSON), ("val", VAL_JSON), ("test", TEST_JSON)]:
    src_dir = f"{DATASET_PATH}/{split if split != 'val' else 'valid'}"
    print(f"Converting {split}...")
    convert_coco_to_yolo(json_f, src_dir,
                         f"{YOLO_DIR}/{split}/images",
                         f"{YOLO_DIR}/{split}/labels")

# ── 2B. Write data.yaml ─────────────────────────────────────────────
YAML_PATH = f"{WORK_DIR}/data.yaml"
with open(YAML_PATH, "w") as f:
    f.write(f"""train: {YOLO_DIR}/train/images
val:   {YOLO_DIR}/val/images
test:  {YOLO_DIR}/test/images

nc: 3
names: [plane, ship, vehicle]
""")
print(f"\n✅ data.yaml saved → {YAML_PATH}")

# ── 2C. THE 15 CANONICAL HYPERPARAMETERS ────────────────────────────
# ⚠️  These exact IDs and values are shared by ALL 5 models (NB1 + NB2).
# ⚠️  batch is overridden per model — all other 14 are locked identical.
# ⚠️  NB3 reads hp_* columns from the CSV and verifies consistency.
#
# Batch sizes used (model-specific, to fit GPU/CPU memory):
#   YOLO11n  → batch = 32  (smallest model, fits more)
#   YOLO11s  → batch = 16
#   YOLO11m  → batch =  4  (reduced from 12 to prevent OOM/crash)
#   RT-DETR-L→ batch =  4  (transformer — heavy)
#   YOLOv8m  → batch =  4  (safe for all hardware)

TRACKED_PARAM_KEYS = [
    "epochs", "imgsz", "batch", "patience", "warmup_epochs",   # P01–P05
    "mosaic", "mixup", "copy_paste", "degrees", "translate",   # P06–P10
    "scale",  "flipud", "fliplr", "hsv_h", "hsv_s",           # P11–P15
]

COMMON_TRAIN_PARAMS = dict(
    # ── Core training ─────────────────────────────────────────────
    epochs        =  50,    # P01  — use 50 for Kaggle time limit; set 100 for final paper
    imgsz         = 640,    # P02
    batch         =   4,    # P03  ← CONSERVATIVE default (overridden per model below)
    patience      =  20,    # P04
    warmup_epochs =   3,    # P05
    # ── Augmentation (aerial imagery specific) ────────────────────
    mosaic        = 1.0,    # P06
    mixup         = 0.1,    # P07
    copy_paste    = 0.1,    # P08
    degrees       = 45.0,   # P09  large rotation valid for overhead view
    translate     = 0.1,    # P10
    scale         = 0.5,    # P11
    flipud        = 0.5,    # P12  vertical flip valid for satellite
    fliplr        = 0.5,    # P13
    hsv_h         = 0.015,  # P14
    hsv_s         = 0.7,    # P15
    # ── Fixed (not in tracked 15, not saved to CSV) ───────────────
    data          = YAML_PATH,
    workers       = 2,
    seed          = 42,
    verbose       = False,
)

descriptions = {
    "epochs":        "Total training epochs",
    "imgsz":         "Input image size (px)",
    "batch":         "Mini-batch size (model-specific)",
    "patience":      "Early-stop patience (epochs)",
    "warmup_epochs": "LR warm-up period (epochs)",
    "mosaic":        "Mosaic aug probability",
    "mixup":         "Mixup aug probability",
    "copy_paste":    "Copy-paste aug probability",
    "degrees":       "Random rotation range (±deg)",
    "translate":     "Random translation (fraction)",
    "scale":         "Random scale range (fraction)",
    "flipud":        "Vertical flip probability",
    "fliplr":        "Horizontal flip probability",
    "hsv_h":         "Hue jitter (fraction)",
    "hsv_s":         "Saturation jitter",
}

print("\n" + "="*65)
print("THE 15 CANONICAL TRAINING HYPERPARAMETERS")
print("="*65)
print(f"  {'ID':<5} {'Parameter':<18} {'Value':<8}  Description")
print("  " + "─"*60)
for i, k in enumerate(TRACKED_PARAM_KEYS, 1):
    print(f"  P{i:02d}   {k:<18} {str(COMMON_TRAIN_PARAMS[k]):<8}  {descriptions[k]}")

print(f"\n  ✅ 15 params locked — shared across ALL 5 models (NB1 + NB2)")
print("  ⚠️  batch (P03) is overridden per model:")
print("        YOLO11n → 32 | YOLO11s → 16 | YOLO11m → 4")
print("        RT-DETR-L → 4 | YOLOv8m → 4")
print("  ⚠️  If you still get OOM, reduce YOLO11n batch to 16 or add amp=True")
print("\n📌 Proceed to Cell 3 — Training + 3-Slice SAHI")


Converting train...
  ✅  1305 converted |    0 missing |    0 degenerate removed
Converting val...
  ✅    54 converted |    0 missing |    0 degenerate removed
Converting test...
  ✅    55 converted |    0 missing |    0 degenerate removed

✅ data.yaml saved → /kaggle/working/data.yaml

THE 15 CANONICAL TRAINING HYPERPARAMETERS
  ID    Parameter          Value     Description
  ────────────────────────────────────────────────────────────
  P01   epochs             50        Total training epochs
  P02   imgsz              640       Input image size (px)
  P03   batch              4         Mini-batch size (model-specific)
  P04   patience           20        Early-stop patience (epochs)
  P05   warmup_epochs      3         LR warm-up period (epochs)
  P06   mosaic             1.0       Mosaic aug probability
  P07   mixup              0.1       Mixup aug probability
  P08   copy_paste         0.1       Copy-paste aug probability
  P09   degrees            45.0      Random rotation rang

## 🚀 Cell 3 — Train YOLO11n / YOLO11s / YOLO11m + SAHI Ablation + Export Results

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — TRAIN + 3-SLICE SAHI + EXPORT
# ═══════════════════════════════════════════════════════════════════════

from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import gc, time

# ── 3-Slice SAHI config (IDENTICAL across all 5 models) ─────────────
SAHI_SIZES   = [480, 320, 160]   # SP01 — three tile sizes
SAHI_OVERLAP = 0.2               # SP02
SAHI_CONF    = 0.25              # SP03
SAHI_N_IMGS  = 50                # SP04

print("="*65)
print("SAHI CONFIGURATION — 3 SLICE SIZES (same for all 5 models)")
print("="*65)
print(f"  SP01  Slice sizes (px)      : {SAHI_SIZES}")
print(f"  SP02  Overlap ratio         : {SAHI_OVERLAP}")
print(f"  SP03  Confidence threshold  : {SAHI_CONF}")
print(f"  SP04  Images per eval       : {SAHI_N_IMGS}")
print("  → Each model produces 3 rows in results CSV (one per slice size)")

# ── Helper: one SAHI ablation at a specific slice size ───────────────
def run_sahi_ablation(model, model_path, test_img_dir, slice_size):
    """
    Runs both standard and SAHI inference on the SAME image set.
    Returns dict with std_avg_det, sahi_avg_det, sahi_gain_pct.
    """
    imgs = [os.path.join(test_img_dir, f)
            for f in sorted(os.listdir(test_img_dir))
            if f.lower().endswith((".jpg",".png"))][:SAHI_N_IMGS]
    if not imgs:
        return {"std_avg_det": 0, "sahi_avg_det": 0, "sahi_gain_pct": 0,
                "sahi_slice": slice_size, "sahi_overlap": SAHI_OVERLAP,
                "sahi_conf": SAHI_CONF, "sahi_n_imgs": 0}

    # Standard inference
    std_counts = []
    for img_path in imgs:
        res = model.predict(img_path, conf=SAHI_CONF, verbose=False, device=device)
        std_counts.append(len(res[0].boxes) if res and res[0].boxes is not None else 0)

    # SAHI sliced inference
    det_model = AutoDetectionModel.from_pretrained(
        model_type           = "yolov8",
        model_path           = model_path,
        confidence_threshold = SAHI_CONF,
        device               = device,
    )
    sahi_counts = []
    for img_path in imgs:
        r = get_sliced_prediction(
            img_path, det_model,
            slice_height         = slice_size,
            slice_width          = slice_size,
            overlap_height_ratio = SAHI_OVERLAP,
            overlap_width_ratio  = SAHI_OVERLAP,
            verbose              = 0,
        )
        sahi_counts.append(len(r.object_prediction_list))
    del det_model
    gc.collect()

    std_avg  = float(np.mean(std_counts))
    sahi_avg = float(np.mean(sahi_counts))
    return {
        "std_avg_det":   round(std_avg,  2),
        "sahi_avg_det":  round(sahi_avg, 2),
        "sahi_gain_pct": round(100*(sahi_avg-std_avg)/max(std_avg, 1e-6), 1),
        "sahi_slice":    slice_size,
        "sahi_overlap":  SAHI_OVERLAP,
        "sahi_conf":     SAHI_CONF,
        "sahi_n_imgs":   len(imgs),
    }

# ── Helper: extract val metrics + FPS ────────────────────────────────
def extract_metrics(results, base_name, train_time_sec, sahi_dict,
                    batch_used, tracked_params):
    best_pt  = os.path.join(results.save_dir, "weights", "best.pt")
    m        = YOLO(best_pt)
    val_res  = m.val(data=YAML_PATH, conf=SAHI_CONF, verbose=False, device=device)
    vd       = val_res.results_dict
    params_m = sum(p.numel() for p in m.model.parameters()) / 1e6

    dummy = torch.zeros(1, 3, 640, 640).to(device)
    t0 = time.time()
    with torch.no_grad():
        for _ in range(30):
            m.model(dummy)
    fps = round(30 / (time.time() - t0), 1)

    p = vd.get("metrics/precision(B)", 0)
    r = vd.get("metrics/recall(B)",    0)
    row = {
        "Model":         f"{base_name}_S{sahi_dict['sahi_slice']}",
        "base_model":    base_name,
        "sahi_slice":    sahi_dict["sahi_slice"],
        "mAP50":         round(vd.get("metrics/mAP50(B)",    0), 4),
        "mAP50_95":      round(vd.get("metrics/mAP50-95(B)", 0), 4),
        "Precision":     round(p, 4),
        "Recall":        round(r, 4),
        "F1":            round(2*p*r / max(p+r, 1e-6), 4),
        "AP50_plane":    round(float(val_res.box.ap50[0]) if hasattr(val_res.box,"ap50") else 0, 4),
        "AP50_ship":     round(float(val_res.box.ap50[1]) if hasattr(val_res.box,"ap50") else 0, 4),
        "AP50_vehicle":  round(float(val_res.box.ap50[2]) if hasattr(val_res.box,"ap50") else 0, 4),
        "FPS":           fps,
        "Params_M":      round(params_m, 2),
        "Train_time_hr": round(train_time_sec / 3600, 2),
        "std_avg_det":   sahi_dict["std_avg_det"],
        "sahi_avg_det":  sahi_dict["sahi_avg_det"],
        "sahi_gain_pct": sahi_dict["sahi_gain_pct"],
        "sahi_overlap":  sahi_dict["sahi_overlap"],
        **{f"hp_{k}": v for k, v in tracked_params.items()},
    }
    del m
    return row, best_pt

# ── YOLO11 models to train ────────────────────────────────────────────
# batch intentionally conservative — reduce further if OOM
MODELS_TO_TRAIN = [
    ("yolo11n.pt", "YOLO11n", 32),  # nano  — lightest
    ("yolo11s.pt", "YOLO11s", 16),  # small
    ("yolo11m.pt", "YOLO11m",  4),  # medium — batch=4 to prevent OOM/crash
]
TEST_IMG_DIR = f"{YOLO_DIR}/test/images"
all_rows     = []

for weights, base_name, batch in MODELS_TO_TRAIN:
    print("\n" + "="*65)
    print(f"🏋️  Training {base_name}  |  batch={batch}  |  device={device}")
    print("="*65)

    # Build training call params
    tracked_for_row = {k: COMMON_TRAIN_PARAMS[k] for k in TRACKED_PARAM_KEYS}
    tracked_for_row["batch"] = batch

    train_call = {k: v for k, v in COMMON_TRAIN_PARAMS.items()
                  if k not in ("conf","iou")}
    train_call.update({
        "batch":   batch,
        "project": f"{WORK_DIR}/runs",
        "name":    base_name,
        "device":  0 if torch.cuda.is_available() else "cpu",
    })

    model    = YOLO(weights)
    t_start  = time.time()
    results  = model.train(**train_call)
    t_train  = time.time() - t_start
    print(f"  ⏱  Training time: {t_train/3600:.2f} hr")

    # ── 3-slice SAHI ablation ──────────────────────────────────────
    best_pt = os.path.join(results.save_dir, "weights", "best.pt")
    for s_size in SAHI_SIZES:
        print(f"  🔪 SAHI ablation  {base_name}  slice={s_size}px ...")
        sahi_info = run_sahi_ablation(model, best_pt, TEST_IMG_DIR, s_size)
        print(f"     Std={sahi_info['std_avg_det']:.2f}  "
              f"SAHI={sahi_info['sahi_avg_det']:.2f}  "
              f"Gain=+{sahi_info['sahi_gain_pct']:.1f}%")
        row, _ = extract_metrics(results, base_name, t_train, sahi_info,
                                 batch, tracked_for_row)
        all_rows.append(row)

    # Save model weight
    out_pt = f"{SAVE_DIR}/{base_name}_best.pt"
    shutil.copy(best_pt, out_pt)
    print(f"  ✅ Model saved → {out_pt}")

    del model, results
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Save NB1 results ─────────────────────────────────────────────────
df_nb1  = pd.DataFrame(all_rows)
NB1_CSV = f"{SAVE_DIR}/nb1_results.csv"
df_nb1.to_csv(NB1_CSV, index=False)

# ── Print before/after SAHI table (all 15 params) ────────────────────
print("\n" + "="*70)
print("15 HYPERPARAMETERS — BEFORE vs AFTER SAHI (inference-only change)")
print("="*70)
print(f"  {'ID':<5} {'Parameter':<18} {'Value':<10} {'SAHI context'}")
print("  " + "─"*72)
sahi_context = {
    "epochs":        "Train only — SAHI runs at inference, not training",
    "imgsz":         "Base res 640 → SAHI tiles into 480/320/160px slices",
    "batch":         "Train only — SAHI processes 1 image at a time",
    "patience":      "Train only — no effect on SAHI",
    "warmup_epochs": "Train only — no effect on SAHI",
    "mosaic":        "Train only — no effect on SAHI",
    "mixup":         "Train only — no effect on SAHI",
    "copy_paste":    "Train only — no effect on SAHI",
    "degrees":       "Train only — no effect on SAHI",
    "translate":     "Train only — no effect on SAHI",
    "scale":         "Train only — no effect on SAHI",
    "flipud":        "Train only — no effect on SAHI",
    "fliplr":        "Train only — no effect on SAHI",
    "hsv_h":         "Train only — no effect on SAHI",
    "hsv_s":         "Train only — no effect on SAHI",
}
for i, k in enumerate(TRACKED_PARAM_KEYS, 1):
    print(f"  P{i:02d}   {k:<18} {str(COMMON_TRAIN_PARAMS[k]):<10} {sahi_context[k]}")

print("\n  SAHI-specific params (separate from the 15 training params):")
print(f"  SP01  sahi_slice_sizes = {SAHI_SIZES}  (3 tile sizes tested per model)")
print(f"  SP02  sahi_overlap     = {SAHI_OVERLAP}")
print(f"  SP03  sahi_conf        = {SAHI_CONF}")
print(f"  SP04  sahi_n_imgs      = {SAHI_N_IMGS}")

# ── Results summary ───────────────────────────────────────────────────
print("\n" + "="*70)
print("NB1 RESULTS SUMMARY (3 rows per model — one per slice size)")
print("="*70)
disp = ["Model","mAP50","mAP50_95","Precision","Recall","F1",
        "FPS","Params_M","std_avg_det","sahi_avg_det","sahi_gain_pct","sahi_slice"]
print(df_nb1[disp].to_string(index=False))
print(f"\n✅ NB1 results ({len(df_nb1)} rows) → {NB1_CSV}")
print("\n📌 Next: Run Notebook 2 (RT-DETR-L + YOLOv8m)")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
SAHI CONFIGURATION — 3 SLICE SIZES (same for all 5 models)
  SP01  Slice sizes (px)      : [480, 320, 160]
  SP02  Overlap ratio         : 0.2
  SP03  Confidence threshold  : 0.25
  SP04  Images per eval       : 50
  → Each model produces 3 rows in results CSV (one per slice size)

🏋️  Training YOLO11n  |  batch=32  |  device=cuda:0
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr

In [4]:
# # ═══════════════════════════════════════════════════════════════════
# # CELL 3 — TRAIN YOLO11m (CLEAN "ONE-BY-ONE" EPOCH OUTPUT)
# # ═══════════════════════════════════════════════════════════════════

# import os
# from ultralytics import YOLO
# from sahi import AutoDetectionModel
# from sahi.predict import get_sliced_prediction
# import gc, time, sys
# import shutil, torch
# import numpy as np
# import pandas as pd

# # ── SAHI config (consistent across ALL 5 models) ─────────────────
# SAHI_SIZES      = [480, 320, 160] # SP01
# SAHI_OVERLAP    = 0.2    # SP02
# SAHI_CONF       = 0.25   # SP03
# SAHI_N_IMGS     = 50     # SP04  images sampled for SAHI eval

# print("="*60)
# print("SAHI CONFIGURATION (YOLO11m ONLY)")
# print("="*60)
# print(f"  SP01  Slice sizes (px)       : {SAHI_SIZES}")
# print(f"  SP02  Overlap ratio          : {SAHI_OVERLAP}")
# print(f"  SP03  Confidence threshold   : {SAHI_CONF}")
# print(f"  SP04  Images tested for SAHI : {SAHI_N_IMGS}")

# # ── Helper: SAHI standard vs sliced comparison ───────────────────
# def run_sahi_ablation(model, model_path, test_img_dir, slice_size):
#     imgs = [os.path.join(test_img_dir, f)
#             for f in sorted(os.listdir(test_img_dir))
#             if f.lower().endswith((".jpg",".png"))][:SAHI_N_IMGS]

#     # ── Standard inference ────────────────────────────────────
#     std_counts = []
#     for img_path in imgs:
#         res = model.predict(img_path, conf=SAHI_CONF, verbose=False,
#                             device=device)
#         n = len(res[0].boxes) if res and res[0].boxes is not None else 0
#         std_counts.append(n)

#     # ── SAHI sliced inference ─────────────────────────────────
#     det_model = AutoDetectionModel.from_pretrained(
#         model_type   = "yolov8",
#         model_path   = model_path,
#         confidence_threshold = SAHI_CONF,
#         device       = device,
#     )
#     sahi_counts = []
#     for img_path in imgs:
#         r = get_sliced_prediction(
#             img_path, det_model,
#             slice_height         = slice_size,
#             slice_width          = slice_size,
#             overlap_height_ratio = SAHI_OVERLAP,
#             overlap_width_ratio  = SAHI_OVERLAP,
#             verbose              = 0,
#         )
#         sahi_counts.append(len(r.object_prediction_list))

#     std_avg  = float(np.mean(std_counts))
#     sahi_avg = float(np.mean(sahi_counts))
#     gain_pct = 100*(sahi_avg - std_avg) / max(std_avg, 1e-6)

#     return {
#         "std_avg_det":  round(std_avg,  2),
#         "sahi_avg_det": round(sahi_avg, 2),
#         "sahi_gain_pct":round(gain_pct, 1),
#         "sahi_slice":   slice_size,
#         "sahi_overlap": SAHI_OVERLAP,
#         "sahi_conf":    SAHI_CONF,
#         "sahi_n_imgs":  len(imgs),
#     }

# # ── Helper: extract full metric row ──────────────────────────────
# def extract_metrics(results, model_name, train_time_sec, sahi_dict,
#                     batch_used, tracked_params):
#     best_pt = os.path.join(results.save_dir, "weights", "best.pt")
#     m = YOLO(best_pt)
#     val_res = m.val(data=YAML_PATH, conf=SAHI_CONF, verbose=False,
#                     device=device)
#     vd = val_res.results_dict
#     params_m = sum(p.numel() for p in m.model.parameters()) / 1e6

#     # FPS on device
#     dummy = torch.zeros(1, 3, 640, 640).to(device)
#     t0 = time.time()
#     with torch.no_grad():
#         for _ in range(30):
#             m.model(dummy)
#     fps = round(30 / (time.time() - t0), 1)

#     p  = vd.get("metrics/precision(B)", 0)
#     r  = vd.get("metrics/recall(B)",    0)
#     row = {
#         "Model":           f"{model_name}_S{sahi_dict['sahi_slice']}",
#         "mAP50":           round(vd.get("metrics/mAP50(B)",    0), 4),
#         "mAP50_95":        round(vd.get("metrics/mAP50-95(B)", 0), 4),
#         "Precision":       round(p, 4),
#         "Recall":          round(r, 4),
#         "F1":              round(2*p*r / max(p+r, 1e-6), 4),
#         "AP50_plane":      round(float(val_res.box.ap50[0]) if hasattr(val_res.box,"ap50") else 0, 4),
#         "AP50_ship":       round(float(val_res.box.ap50[1]) if hasattr(val_res.box,"ap50") else 0, 4),
#         "AP50_vehicle":    round(float(val_res.box.ap50[2]) if hasattr(val_res.box,"ap50") else 0, 4),
#         "FPS":             fps,
#         "Params_M":        round(params_m, 2),
#         "Train_time_hr":   round(train_time_sec / 3600, 2),
#         "std_avg_det":     sahi_dict["std_avg_det"],
#         "sahi_avg_det":    sahi_dict["sahi_avg_det"],
#         "sahi_gain_pct":   sahi_dict["sahi_gain_pct"],
#         "sahi_slice":      sahi_dict["sahi_slice"],
#         "sahi_overlap":    sahi_dict["sahi_overlap"],
#         **{f"hp_{k}": v for k, v in tracked_params.items()},
#     }
#     return row, best_pt


# # ── CUSTOM CALLBACK TO PRINT ONE BY ONE ───────────────────────────
# def on_train_epoch_end(trainer):
#     """Prints a clean, single line when an epoch finishes."""
#     epoch = trainer.epoch + 1
#     total_epochs = trainer.epochs
#     print(f"  👉 Completed Epoch {epoch}/{total_epochs}", flush=True)


# # ── MODELS TO TRAIN ───────────────────────────────────────────────
# MODELS_TO_TRAIN = [
#     ("/kaggle/working/yolo11m.pt", "YOLO11m", 4), # Local file, Batch 4 ONLY!
# ]
# TEST_IMG_DIR = f"{YOLO_DIR}/test/images"
# all_rows     = []

# for weights, name, batch in MODELS_TO_TRAIN:
#     print("\n" + "="*60)
#     print(f"🏋️  Training {name}  |  batch={batch}  |  device={device}")
#     print("="*60)

#     # verbose=False hides the browser-crashing progress bar, workers=0 prevents freeze
#     train_params_used = {**COMMON_TRAIN_PARAMS, "batch": batch, "workers": 0, "verbose": False}
    
#     tracked_for_row = {k: train_params_used.get(k, 0) for k in TRACKED_PARAM_KEYS}
#     tracked_for_row["batch"] = batch  # ensure actual batch is stored

#     train_call_params = {k: v for k, v in train_params_used.items()
#                          if k not in ("conf","iou")}
#     train_call_params.update({"project": f"{WORK_DIR}/runs", "name": name,
#                                "device": 0 if torch.cuda.is_available() else "cpu"})

#     model = YOLO(weights)
    
#     # Attach our custom printer to the model!
#     model.add_callback("on_train_epoch_end", on_train_epoch_end)
    
#     t_start  = time.time()
#     print(f"  ⏳ Training started! You will see each epoch print one-by-one below...\n")
    
#     # Run training
#     results  = model.train(**train_call_params)
            
#     t_elapsed = time.time() - t_start
#     print(f"\n  ✅ Training Finished! Time: {t_elapsed/3600:.2f} hr")

#     # SAHI ablation (standard vs sliced)
#     best_pt = os.path.join(results.save_dir, "weights", "best.pt")
    
#     for s_size in SAHI_SIZES:
#         print(f"  🔪 SAHI ablation on {name} | Slice: {s_size}px...")
#         sahi_info = run_sahi_ablation(model, best_pt, TEST_IMG_DIR, s_size)
#         print(f"     Standard avg det : {sahi_info['std_avg_det']}")
#         print(f"     SAHI avg det     : {sahi_info['sahi_avg_det']}")
#         print(f"     SAHI gain        : +{sahi_info['sahi_gain_pct']}%")

#         row, _ = extract_metrics(results, name, t_elapsed, sahi_info,
#                                  batch, tracked_for_row)
#         all_rows.append(row)

#     # Save model weight
#     out_pt = f"{SAVE_DIR}/{name}_best.pt"
#     shutil.copy(best_pt, out_pt)
#     print(f"  ✅ Saved → {out_pt}")

#     del model
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# # ── Save Results JUST for YOLO11m ──────────────────────────────────
# df_nb1_m = pd.DataFrame(all_rows)
# NB1_CSV  = f"{SAVE_DIR}/yolo11m_results.csv"
# df_nb1_m.to_csv(NB1_CSV, index=False)

# # ── Print the 15-param before/after SAHI table ───────────────────
# print("\n" + "="*70)
# print("15 HYPERPARAMETERS — BEFORE vs AFTER SAHI ABLATION")
# print("="*70)
# print(f"  {'ID':<4} {'Parameter':<18} {'Value (all models)':<22} {'Role in SAHI context'}")
# print("-"*80)
# sahi_context = {
#     "epochs":        "Unaffected — SAHI is inference-only",
#     "imgsz":         "Base res; SAHI slices this to 320px tiles",
#     "batch":         "Training only; SAHI uses batch=1",
#     "patience":      "Training only; no SAHI effect",
#     "warmup_epochs": "Training only; no SAHI effect",
#     "mosaic":        "Training only; no SAHI effect",
#     "mixup":         "Training only; no SAHI effect",
#     "copy_paste":    "Training only; no SAHI effect",
#     "degrees":       "Training only; no SAHI effect",
#     "translate":     "Training only; no SAHI effect",
#     "scale":         "Training only; no SAHI effect",
#     "flipud":        "Training only; no SAHI effect",
#     "fliplr":        "Training only; no SAHI effect",
#     "hsv_h":         "Training only; no SAHI effect",
#     "hsv_s":         "Training only; no SAHI effect",
# }
# for i, k in enumerate(TRACKED_PARAM_KEYS, 1):
#     val = str(COMMON_TRAIN_PARAMS.get(k, ""))
#     print(f"  P{i:02d}  {k:<18} {val:<22} {sahi_context[k]}")

# print("\n  SAHI-specific parameters (additional, consistent across all models):")
# print(f"  SP01  sahi_slice_size   = {SAHI_SIZES}  (px — tile sizes)")
# print(f"  SP02  sahi_overlap      = {SAHI_OVERLAP}  (tile overlap ratio)")
# print(f"  SP03  sahi_conf         = {SAHI_CONF}  (confidence threshold)")
# print(f"  SP04  sahi_n_imgs       = {SAHI_N_IMGS}  (images evaluated)")

# # ── Print results summary ─────────────────────────────────────────
# print("\n" + "="*70)
# print("NB1 RESULTS SUMMARY (YOLO11m ONLY)")
# print("="*70)
# disp_cols = ["Model","mAP50","mAP50_95","Precision","Recall","F1",
#              "AP50_plane","AP50_ship","AP50_vehicle",
#              "FPS","Params_M","Train_time_hr",
#              "std_avg_det","sahi_avg_det","sahi_gain_pct"]
# print(df_nb1_m[disp_cols].to_string(index=False))
# print(f"\n✅ YOLO11m results saved → {NB1_CSV}")
# print("\n📌 Next: Run Notebook 2 (RT-DETR-L + YOLOv8m)")
# print("📌 Then: Load both CSVs into Notebook 3 for final evaluation.")